# BASF Chemicals Demand Data Generator

Generates **synthetic chemical product demand data** with domain-specific patterns:
- Products modeled after real BASF chemical families (engineering plastics, MDI, superabsorbents)
- Seasonal demand cycles tied to plant turnaround schedules (Q2/Q4 dips)
- Exogenous regressors: feedstock price index, plant utilization rate, regulatory flag
- Monthly resolution, 60-month history per SKU

Output table: `mmf_demo_aym.m4_advanced.basf_chemicals_train`

Use with `forecasting_conf_chemicals.yaml` for the MMF pipeline.

In [0]:
import numpy as np
import pandas as pd

np.random.seed(42)

# BASF-style chemical product families
PRODUCTS = {
    # Engineering Plastics
    "Ultramid_B3S": {"base": 2400, "noise": 180, "trend": 8, "seasonal_amp": 0.12},
    "Ultramid_A3WG6": {"base": 1800, "noise": 150, "trend": 5, "seasonal_amp": 0.10},
    "Ultraform_N2320": {"base": 950, "noise": 80, "trend": 3, "seasonal_amp": 0.08},
    # MDI / Isocyanates
    "Lupranate_M20S": {"base": 5200, "noise": 400, "trend": 15, "seasonal_amp": 0.15},
    "Lupranate_MI": {"base": 3100, "noise": 250, "trend": 10, "seasonal_amp": 0.13},
    # Polyols
    "Pluracol_1509": {"base": 2800, "noise": 200, "trend": 6, "seasonal_amp": 0.09},
    "Pluracol_2090": {"base": 1600, "noise": 130, "trend": 4, "seasonal_amp": 0.11},
    # Superabsorbents
    "Hysorb_B7065": {"base": 4100, "noise": 320, "trend": 12, "seasonal_amp": 0.07},
    # Catalysts
    "FCC_Catalyst_DMS": {"base": 680, "noise": 55, "trend": 2, "seasonal_amp": 0.20},
    # Coatings
    "Acronal_S728": {"base": 3500, "noise": 280, "trend": 9, "seasonal_amp": 0.14},
    "Acronal_DS2373": {"base": 2200, "noise": 170, "trend": 7, "seasonal_amp": 0.12},
    # Intermediates
    "Caprolactam_Grade1": {"base": 6800, "noise": 500, "trend": 20, "seasonal_amp": 0.16},
}

N_MONTHS = 60
start = pd.Timestamp("2020-01-31")
dates = pd.date_range(start=start, periods=N_MONTHS, freq="ME")

rows = []
for product, params in PRODUCTS.items():
    for i, dt in enumerate(dates):
        month = dt.month
        # Seasonal: dips in Q2 (April turnaround) and Q4 (Oct maintenance)
        seasonal = 1.0 + params["seasonal_amp"] * np.sin(2 * np.pi * (month - 3) / 12)
        # Plant turnaround dip in April and October
        turnaround = 0.85 if month in (4, 10) else 1.0
        # Trend
        trend = params["trend"] * i
        # Demand
        demand = (params["base"] + trend) * seasonal * turnaround
        demand += np.random.normal(0, params["noise"])
        demand = max(0, demand)

        # Exogenous regressors
        # Feedstock price index (correlated with oil, monthly variation)
        feedstock_idx = 100 + 20 * np.sin(2 * np.pi * i / 24) + np.random.normal(0, 5)
        # Plant utilization rate (0.7-0.98, dips during turnarounds)
        util_base = 0.92 if month not in (4, 10) else 0.78
        plant_util = min(0.99, max(0.60, util_base + np.random.normal(0, 0.03)))
        # REACH/regulatory compliance flag (1=compliant, 0=restriction period)
        regulatory = 0 if (product.startswith("Lupranate") and month in (1, 7)) else 1

        rows.append({
            "ds": dt,
            "unique_id": product,
            "y": round(demand, 2),
            "feedstock_price_index": round(feedstock_idx, 2),
            "plant_utilization_rate": round(plant_util, 4),
            "regulatory_compliance": regulatory,
        })

chemicals_df = pd.DataFrame(rows)
print(f"Generated {len(chemicals_df)} rows for {len(PRODUCTS)} chemical products")
print(f"Date range: {chemicals_df['ds'].min()} to {chemicals_df['ds'].max()}")
display(chemicals_df.head(12))

Generated 720 rows for 12 chemical products
Date range: 2020-01-31 00:00:00 to 2024-12-31 00:00:00
        ds    unique_id       y  feedstock_price_index  plant_utilization_rate  regulatory_compliance
2020-01-31 Ultramid_B3S 2239.99                  99.31                  0.9394                      1
2020-02-29 Ultramid_B3S 2537.67                 104.01                  0.9130                      1
2020-03-31 Ultramid_B3S 2700.26                 113.84                  0.9059                      1
2020-04-30 Ultramid_B3S 2281.68                 111.83                  0.7660                      1
2020-05-31 Ultramid_B3S 2728.29                 107.75                  0.8683                      1
2020-06-30 Ultramid_B3S 2631.59                 114.25                  0.9294                      1
2020-07-31 Ultramid_B3S 2538.96                 112.94                  0.9640                      1
2020-08-31 Ultramid_B3S 2562.72                 119.66                  0.8773       

In [0]:
catalog = "mmf_demo_aym"
schema = "m4_advanced"
table = "basf_chemicals_train"

spark_df = spark.createDataFrame(chemicals_df)
spark_df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table}")

count = spark.table(f"{catalog}.{schema}.{table}").count()
print(f"Written {count} rows to {catalog}.{schema}.{table}")

# Add table comment for governance
spark.sql(f"""
    COMMENT ON TABLE {catalog}.{schema}.{table} IS
    'Synthetic BASF-style chemical product demand data with domain-specific "
    "seasonal patterns (plant turnarounds), exogenous regressors (feedstock price, "
    "plant utilization, regulatory compliance), and 12 product families across "
    "engineering plastics, MDI, polyols, superabsorbents, catalysts, and coatings.'
""")
print("Table comment added.")

Written 720 rows to mmf_demo_aym.m4_advanced.basf_chemicals_train
Table comment added.
Column comments added.
